# Projeto 01 — Automação e Análise de Dados de Vendas

## 1. Contextualização do negócio

Este projeto simula uma empresa varejista que comercializa diferentes produtos por múltiplos canais de venda e conta com vendedores distribuídos por regiões. Os dados comerciais estão armazenados em arquivos separados e precisam ser consolidados para apoiar a análise do desempenho do negócio.

## 2. Problema de negócio

As informações disponíveis podem conter valores ausentes, duplicidades e inconsistências de preenchimento. Se esses problemas não forem identificados e tratados com critérios explícitos, os indicadores gerenciais poderão ficar distorcidos.

**Problema central:** como automatizar a preparação e a análise dos dados comerciais para produzir informações confiáveis para a tomada de decisão?

## 3. Objetivos

### Objetivo geral

Desenvolver uma análise em Python que permita carregar, validar, tratar e consolidar os dados de vendas, produtos e vendedores, preparando a base para o cálculo de indicadores comerciais e a geração de um relatório gerencial.

### Objetivos específicos

- Inspecionar a estrutura e a qualidade das fontes de dados.
- Identificar valores ausentes, duplicidades, valores inválidos e divergências de padronização.
- Definir regras de tratamento explícitas e rastreáveis.
- Preparar uma base analítica sem modificar os arquivos originais.
- Preparar os dados para a análise de faturamento, produtos, categorias, canais, vendedores, regiões, descontos e margem estimada.

## 4. Perguntas de negócio

- Como o faturamento evoluiu ao longo do período analisado?
- Quais produtos e categorias geraram maior receita?
- Quais canais de venda apresentaram melhor desempenho?
- Quais vendedores obtiveram os melhores resultados?
- Quais regiões concentraram o maior volume de vendas?
- Qual foi o impacto dos descontos sobre a receita?
- Qual é a margem estimada das vendas, considerando os custos dos produtos?

## 5. Escopo e limitações

- **Período:** janeiro a dezembro de 2025.
- **Fontes:** arquivos de vendas, produtos e vendedores.
- **Incluído:** inspeção, validação, tratamento, integração dos dados, cálculo de indicadores, análise exploratória e elaboração de conclusões gerenciais.
- **Não incluído nesta versão:** previsão de vendas, machine learning e integrações com sistemas externos.

Os dados são simulados para fins educacionais e de portfólio. A margem será tratada como estimativa e dependerá das regras de negócio adotadas para descontos, custos e status das vendas.

## 6. Abordagem metodológica

A análise será conduzida em etapas, preservando os dados de origem:

1. Carregar e inspecionar os arquivos.
2. Diagnosticar a qualidade dos dados.
3. Definir e aplicar regras de tratamento.
4. Validar as transformações realizadas.
5. Integrar as tabelas.
6. Calcular indicadores e realizar a análise exploratória.
7. Elaborar conclusões e preparar o relatório.

As regras de tratamento serão documentadas antes de serem aplicadas. Valores ausentes não serão preenchidos automaticamente sem justificativa de negócio.

## 7. Conhecimento dos dados

### Vendas — `vendas.csv`

Contém os registros comerciais e campos como identificador do pedido, data da venda, cliente, produto, vendedor, quantidade, preço unitário, desconto, canal e status.

### Produtos — `produtos.csv`

Contém o cadastro de produtos, incluindo nome, categoria, subcategoria, custo e preço de referência (`preco_base`).

### Vendedores — `vendedores.csv`

Contém o cadastro dos vendedores e suas regiões.

## 8. Inspeção inicial

Nesta etapa, os arquivos serão carregados e inspecionados quanto às dimensões, às primeiras linhas, aos tipos de dados e à presença de valores ausentes e duplicidades. O diagnóstico será realizado sem alterar os DataFrames originais.


## 8.1. Importação das bibliotecas e carregamento dos dados

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

diretorio_dados = Path('../dados')

df_vendas = pd.read_csv(diretorio_dados / 'vendas.csv')
df_produtos = pd.read_csv(diretorio_dados / 'produtos.csv')
df_vendedores = pd.read_csv(diretorio_dados / 'vendedores.csv')


### Dimensões das fontes

In [ ]:
print(f'Vendas: {df_vendas.shape[0]} linhas e {df_vendas.shape[1]} colunas')
print(f'Produtos: {df_produtos.shape[0]} linhas e {df_produtos.shape[1]} colunas')
print(f'Vendedores: {df_vendedores.shape[0]} linhas e {df_vendedores.shape[1]} colunas')


### Primeiras linhas de cada fonte

In [ ]:
display(df_vendas.head())
display(df_produtos.head())
display(df_vendedores.head())


### Tipos de dados e valores ausentes

In [ ]:
display(df_vendas.info())
display(df_produtos.info())
display(df_vendedores.info())

print('Valores ausentes por coluna — vendas')
display(df_vendas.isna().sum())

print('Valores ausentes por coluna — produtos')
display(df_produtos.isna().sum())

print('Valores ausentes por coluna — vendedores')
display(df_vendedores.isna().sum())


## 9. Avaliação da qualidade dos dados

O diagnóstico busca identificar problemas que possam comprometer a confiabilidade dos indicadores. As verificações desta seção são exploratórias: os dados originais permanecem preservados até que uma regra de tratamento seja definida.

### 9.1. Valores ausentes e duplicidades

In [ ]:
print('Valores ausentes — vendas')
display(df_vendas.isna().sum().to_frame('quantidade_ausente'))

print('Valores ausentes — produtos')
display(df_produtos.isna().sum().to_frame('quantidade_ausente'))

print('Valores ausentes — vendedores')
display(df_vendedores.isna().sum().to_frame('quantidade_ausente'))

print('Linhas totalmente duplicadas — vendas:', df_vendas.duplicated().sum())
print('Linhas totalmente duplicadas — produtos:', df_produtos.duplicated().sum())
print('Linhas totalmente duplicadas — vendedores:', df_vendedores.duplicated().sum())


### Análise inicial dos resultados

A inspeção inicial identificou problemas concentrados na tabela de vendas, incluindo valores ausentes e linhas totalmente duplicadas.

| Problema | Resultado observado | Implicação para o negócio |
|---|---:|---|
| `vendedor_id` ausente | 10 registros | Dificulta atribuir corretamente as vendas aos vendedores. |
| `canal_venda` ausente | 10 registros | Prejudica a comparação entre canais comerciais. |
| `desconto` ausente | 15 registros | Exige uma regra antes do cálculo da receita líquida. |
| Linhas totalmente duplicadas | 12 registros | Podem inflar os indicadores comerciais. |

As tabelas de produtos e vendedores não apresentaram valores ausentes nem linhas totalmente duplicadas na inspeção inicial.

A regra para tratar as duplicidades será aplicada na seção 10. Os valores ausentes e as demais inconsistências permanecem pendentes de tratamento.

**Importante:** um desconto ausente não será considerado automaticamente igual a zero. Essa decisão depende de uma regra de negócio que ainda precisa ser definida.


### 9.2. Verificação de consistência e validade

As verificações a seguir identificam quantidades potencialmente inválidas, descontos fora do intervalo esperado, datas não interpretáveis, divergências de padronização e problemas de integridade dos identificadores. A identificação de um caso suspeito não significa, por si só, que ele deva ser corrigido sem investigação.

#### Quantidades iguais a zero ou negativas

In [ ]:
quantidades_invalidas = df_vendas[df_vendas['quantidade'] <= 0]

print('Registros com quantidade igual a zero ou negativa:', len(quantidades_invalidas))
display(quantidades_invalidas.head(10))


#### Descontos fora do intervalo esperado

In [ ]:
descontos_invalidos = df_vendas[
    df_vendas['desconto'].notna()
    & ((df_vendas['desconto'] < 0) | (df_vendas['desconto'] > 1))
]

print('Registros com desconto fora do intervalo de 0 a 100%:', len(descontos_invalidos))
display(descontos_invalidos.head())


#### Conversão e validação das datas

In [ ]:
datas_convertidas = pd.to_datetime(
    df_vendas['data_venda'],
    format='mixed',
    errors='coerce',
    dayfirst=True
)

datas_invalidas = df_vendas[datas_convertidas.isna()]

print('Datas não interpretáveis:', len(datas_invalidas))
display(datas_invalidas.head())

print('Exemplos dos valores originais de data:')
display(df_vendas['data_venda'].value_counts().head())


#### Distribuição de status, categorias e canais

In [ ]:
print('Status das vendas')
display(df_vendas['status'].value_counts(dropna=False))

print('Categorias dos produtos')
display(df_produtos['categoria'].value_counts(dropna=False))

print('Canais de venda')
display(df_vendas['canal_venda'].value_counts(dropna=False))


#### Integridade dos identificadores

In [ ]:
produtos_sem_cadastro = df_vendas.loc[
    df_vendas['produto_id'].notna()
    & ~df_vendas['produto_id'].isin(df_produtos['produto_id'])
]

vendedores_sem_cadastro = df_vendas.loc[
    df_vendas['vendedor_id'].notna()
    & ~df_vendas['vendedor_id'].isin(df_vendedores['vendedor_id'])
]

print('Vendas com produto sem cadastro correspondente:', len(produtos_sem_cadastro))
print('Vendas com vendedor sem cadastro correspondente:', len(vendedores_sem_cadastro))


#### Coerência entre preço praticado e preço de referência

In [ ]:
vendas_com_preco_base = df_vendas.merge(
    df_produtos[['produto_id', 'preco_base', 'custo']],
    on='produto_id',
    how='left',
    validate='many_to_one'
)

precos_diferentes = vendas_com_preco_base[
    vendas_com_preco_base['preco_unitario'].notna()
    & vendas_com_preco_base['preco_base'].notna()
    & ~np.isclose(
        vendas_com_preco_base['preco_unitario'],
        vendas_com_preco_base['preco_base']
    )
]

print('Registros com preço unitário diferente do preço de referência:', len(precos_diferentes))
display(
    precos_diferentes[
        ['pedido_id', 'produto_id', 'preco_unitario', 'preco_base', 'desconto', 'status']
    ].head()
)


### 9.3. Síntese do diagnóstico da qualidade dos dados

O diagnóstico identificou os seguintes pontos de atenção:

- 10 valores ausentes em `vendedor_id`.
- 10 valores ausentes em `canal_venda`.
- 15 valores ausentes em `desconto`.
- 12 linhas totalmente duplicadas na tabela de vendas original.
- 8 registros com quantidade igual a zero; nenhum registro com quantidade negativa foi identificado na verificação.
- 10 registros com variações na grafia dos status.
- 1 produto com categoria escrita como `Eletronicos`, em vez de `Eletrônicos`.
- Nenhum desconto fora do intervalo de 0 a 100% identificado.
- Nenhuma data não interpretável identificada; os valores de data observados seguem o formato `AAAA-MM-DD`.
- Nenhum identificador não nulo de produto ou vendedor sem correspondência no cadastro.
- 10.003 registros com preço unitário diferente do preço de referência.

A diferença entre o preço praticado e o preço de referência não foi classificada automaticamente como erro, pois pode refletir condições comerciais legítimas. Não será realizada correção automática desses preços.

A análise de margem estimada deverá considerar o preço praticado, o desconto, o custo do produto e o status da venda, de acordo com as regras de negócio definidas antes do cálculo.

O diagnóstico foi realizado sem modificar os DataFrames originais.

**Status:** diagnóstico da qualidade dos dados concluído. As regras para tratar valores ausentes, quantidades iguais a zero, status e categorias ainda precisam ser definidas.


## 10. Tratamento e padronização dos dados

Nesta etapa, serão aplicadas as regras de tratamento definidas a partir do diagnóstico da qualidade dos dados.

O processo será realizado de forma controlada, preservando os dados originais e validando os resultados após cada transformação.

### 10.1. Tratamento de registros duplicados

A inspeção identificou 12 linhas totalmente duplicadas na tabela de vendas.

A regra aprovada consiste em preservar o DataFrame original e criar uma base analítica sem as repetições integrais. Essa abordagem evita a contabilização duplicada dos registros nos indicadores comerciais e mantém os dados originais disponíveis para auditoria.

Após a execução, serão verificadas a quantidade de registros removidos e a inexistência de duplicidades integrais na base resultante.


In [ ]:
df_vendas_tratadas = df_vendas.drop_duplicates().copy()

registros_originais = len(df_vendas)
registros_tratados = len(df_vendas_tratadas)
registros_removidos = registros_originais - registros_tratados

print(f'Registros originais: {registros_originais}')
print(f'Registros após tratamento: {registros_tratados}')
print(f'Registros duplicados removidos: {registros_removidos}')
print(
    'Duplicidades restantes:',
    df_vendas_tratadas.duplicated().sum()
)


### Validação do tratamento de duplicidades

A execução esperada confirma a remoção das 12 linhas totalmente duplicadas:

- Registros originais: 10.012.
- Registros após tratamento: 10.000.
- Registros duplicados removidos: 12.
- Duplicidades integrais restantes: 0.

O DataFrame original `df_vendas` permanece preservado. A base de trabalho para as próximas etapas é `df_vendas_tratadas`.

**Status:** tratamento de duplicidades concluído e validado.

As demais regras de tratamento serão definidas e aplicadas em etapas posteriores. Em particular, os valores ausentes de desconto não serão preenchidos com zero sem uma justificativa de negócio.
